# 多模型性能对比分析

对比多个模型的per-class性能指标，包括baseline
- F1 Score per Class  
- True vs Predicted Class Prevalence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import nibabel as nib

# 添加父目录到路径，以便导入visualization_toolkit
parent_dir = Path().absolute().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# 导入per-class分析工具
from visualization_toolkit.per_class_analyzer import calculate_per_class_metrics_detailed

# 设置中文字体支持（如果需要）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# 设置seaborn样式
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✅ 导入库完成")

## 1. 配置模型路径和名称

In [ ]:
# ========================================
# 配置区域：修改这里来设置你要对比的模型
# ========================================

# 模型配置：支持两种格式
# 1. CSV路径字符串: 'path/to/per_class_detailed_metrics.csv'
# 2. (softmax_path, labels_path) 元组: 会自动从softmax文件生成分析

models = {
    # Baseline: 从softmax文件生成分析
    'Baseline': (
        '../results/test_softmax_3d_FOR_016_20250204_reproducibility_bg_excl_20250827_154806.nii.gz',
        # ⚠️ 修改下面的labels文件路径为你的实际路径
        '/Users/jannik/KAN-Brain-Single-Voxel-Segmentaion/D_proj_analysis/FOR_016/balanced_output/balanced_labels_3d10000.nii.gz'
    ),
    
    # KAN和DeepMLP: 使用已生成的CSV
    # 取消注释并修改路径
    # 'KAN': 'results/per_class_analysis_kan_bg_excl_TIMESTAMP/per_class_detailed_metrics.csv',
    'DeepMLP': 'results/per_class_analysis_deep_mlp_bg_excl_20251103_200903/per_class_detailed_metrics.csv',
}

# 颜色配置：每个模型对应一个颜色
model_colors = {
    'Baseline': '#1f77b4',  # 蓝色
    'KAN': '#ff7f0e',       # 橙色
    'DeepMLP': '#2ca02c',   # 绿色
}

# FreeSurfer标签映射文件路径
label_mapping_file = '/Users/jannik/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi (1).xlsx'

print(f"📋 配置了 {len(models)} 个模型:")
for name, path in models.items():
    if isinstance(path, tuple):
        print(f"  - {name} (颜色: {model_colors[name]}) - 从softmax生成")
    else:
        print(f"  - {name} (颜色: {model_colors[name]}) - 从CSV加载")

## 2. 加载FreeSurfer标签映射

In [ ]:
# 读取Excel文件
try:
    label_df = pd.read_excel(label_mapping_file)
    
    # 检查列名并显示
    print(f"📊 Excel文件列名: {label_df.columns.tolist()}")
    
    # 假设列名为 'one_hot_loc_alex_label' 和 'tissue_name'
    # 如果列名不同，请修改这里
    label_col = 'one_hot_loc_alex_label'  # 数字标签列
    name_col = 'tissue_name'              # 组织名称列
    
    # 创建映射字典：{class_id: tissue_name}
    label_mapping = dict(zip(label_df[label_col], label_df[name_col]))
    
    print(f"✅ 加载了 {len(label_mapping)} 个标签映射")
    print(f"\n示例映射:")
    for i, (k, v) in enumerate(list(label_mapping.items())[:5]):
        print(f"  {k} -> {v}")
    
except Exception as e:
    print(f"⚠️ 加载标签映射失败: {e}")
    print("将使用原始class_id作为标签")
    label_mapping = {}

## 3. 辅助函数：从Softmax生成per-class分析

In [ ]:
def generate_perclass_from_softmax(softmax_path, labels_path):
    """
    从softmax文件和labels文件生成per-class分析数据
    
    Returns:
        DataFrame: per-class metrics
    """
    print(f"  📂 加载Softmax: {Path(softmax_path).name}")
    print(f"  📂 加载Labels: {Path(labels_path).name}")
    
    # 1. 加载softmax
    softmax_nii = nib.load(softmax_path)
    softmax = softmax_nii.get_fdata()
    print(f"  Softmax形状: {softmax.shape}")
    
    # 2. 加载labels
    labels_nii = nib.load(labels_path)
    labels = labels_nii.get_fdata()
    print(f"  Labels形状: {labels.shape}")
    
    # 3. 计算predictions
    predictions = np.argmax(softmax, axis=-1)
    print(f"  Predictions形状: {predictions.shape}")
    
    # 4. Flatten
    labels_flat = labels.flatten()
    predictions_flat = predictions.flatten()
    print(f"  总体素数: {len(labels_flat):,}")
    
    # 5. 计算per-class metrics
    print("  🧮 计算per-class metrics...")
    metrics_dict = calculate_per_class_metrics_detailed(
        y_true=labels_flat,
        y_pred=predictions_flat
    )
    
    # 6. 转换为DataFrame
    df = pd.DataFrame(metrics_dict['per_class_metrics'])
    
    print(f"  ✅ 分析完成: {len(df)} 个类别")
    print(f"  Macro F1: {metrics_dict['overall_metrics']['macro_f1']:.4f}")
    
    return df

print("✅ 辅助函数定义完成")

## 4. 加载所有模型的数据

In [ ]:
# 加载所有模型的数据
model_data = {}

for model_name, path_config in models.items():
    print(f"\n📊 处理模型: {model_name}")
    
    try:
        # 判断是CSV路径还是(softmax, labels)元组
        if isinstance(path_config, tuple):
            # 从softmax生成
            softmax_path, labels_path = path_config
            
            # 检查文件是否存在
            if not Path(softmax_path).exists():
                print(f"  ❌ Softmax文件不存在: {softmax_path}")
                continue
            if not Path(labels_path).exists():
                print(f"  ❌ Labels文件不存在: {labels_path}")
                print(f"  ⚠️  请修改配置中的labels文件路径")
                continue
            
            df = generate_perclass_from_softmax(softmax_path, labels_path)
        else:
            # 从CSV加载
            csv_path = path_config
            print(f"  📂 加载CSV: {Path(csv_path).name}")
            df = pd.read_csv(csv_path)
            print(f"  ✅ 加载了 {len(df)} 个类别")
        
        # 添加组织名称列
        if label_mapping:
            df['tissue_name'] = df['class_id'].map(label_mapping)
            # 如果没有映射到，使用class_id
            df['tissue_name'] = df['tissue_name'].fillna('Class_' + df['class_id'].astype(str))
        else:
            df['tissue_name'] = 'Class_' + df['class_id'].astype(str)
        
        model_data[model_name] = df
        print(f"  ✅ {model_name}: 成功加载")
        
    except Exception as e:
        print(f"  ❌ {model_name}: 加载失败 - {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"📊 成功加载 {len(model_data)} 个模型的数据")
print(f"{'='*60}")

## 5. 数据预览

In [ ]:
# 查看每个模型的数据示例
if model_data:
    for model_name, df in model_data.items():
        print(f"\n📋 {model_name} 数据预览:")
        print(df[['class_id', 'tissue_name', 'f1_score', 'support', 'prevalence', 'predicted_prevalence']].head(10))
        print(f"\n统计:")
        print(f"  平均F1: {df['f1_score'].mean():.4f}")
        print(f"  中位F1: {df['f1_score'].median():.4f}")
        print(f"  F1>0.8的类别数: {(df['f1_score'] >= 0.8).sum()}")

## 6. 图表1：F1 Score per Class 对比

In [ ]:
# 准备数据：找到所有模型共同的类别
if len(model_data) > 0:
    # 获取所有模型的class_id集合的交集
    common_classes = set(model_data[list(model_data.keys())[0]]['class_id'])
    for df in model_data.values():
        common_classes = common_classes.intersection(set(df['class_id']))
    
    common_classes = sorted(list(common_classes))
    print(f"📊 共同类别数: {len(common_classes)}")
    
    # 准备绘图数据
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 16))
    fig.suptitle('F1 Score per Class - Model Comparison', fontsize=18, fontweight='bold', y=0.995)
    
    # ========================================
    # 上图：按F1分数排序（降序）
    # ========================================
    
    # 计算平均F1分数用于排序
    avg_f1 = {}
    for class_id in common_classes:
        f1_values = []
        for model_name, df in model_data.items():
            f1 = df[df['class_id'] == class_id]['f1_score'].values[0]
            f1_values.append(f1)
        avg_f1[class_id] = np.mean(f1_values)
    
    # 按平均F1排序
    sorted_classes = sorted(common_classes, key=lambda x: avg_f1[x], reverse=True)
    
    # 获取组织名称
    tissue_names = []
    for class_id in sorted_classes:
        tissue_name = model_data[list(model_data.keys())[0]][model_data[list(model_data.keys())[0]]['class_id'] == class_id]['tissue_name'].values[0]
        tissue_names.append(tissue_name)
    
    # 绘制柱状图
    x = np.arange(len(sorted_classes))
    width = 0.8 / len(model_data)  # 柱子宽度
    
    for i, (model_name, df) in enumerate(model_data.items()):
        f1_scores = []
        for class_id in sorted_classes:
            f1 = df[df['class_id'] == class_id]['f1_score'].values[0]
            f1_scores.append(f1)
        
        offset = (i - len(model_data)/2 + 0.5) * width
        ax1.bar(x + offset, f1_scores, width, 
               label=model_name, 
               color=model_colors[model_name], 
               alpha=0.8,
               edgecolor='black',
               linewidth=0.5)
    
    ax1.set_xlabel('Class (sorted by average F1 score)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax1.set_title('F1 Score Comparison - Sorted by Performance', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in zip(tissue_names, sorted_classes)], 
                        rotation=90, ha='center', fontsize=8)
    ax1.set_ylim(0, 1)
    ax1.legend(loc='upper right', fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Excellent (0.8)')
    ax1.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Good (0.6)')
    
    # ========================================
    # 下图：按class_id排序
    # ========================================
    
    sorted_by_id = sorted(common_classes)
    tissue_names_by_id = []
    for class_id in sorted_by_id:
        tissue_name = model_data[list(model_data.keys())[0]][model_data[list(model_data.keys())[0]]['class_id'] == class_id]['tissue_name'].values[0]
        tissue_names_by_id.append(tissue_name)
    
    x2 = np.arange(len(sorted_by_id))
    
    for i, (model_name, df) in enumerate(model_data.items()):
        f1_scores = []
        for class_id in sorted_by_id:
            f1 = df[df['class_id'] == class_id]['f1_score'].values[0]
            f1_scores.append(f1)
        
        offset = (i - len(model_data)/2 + 0.5) * width
        ax2.bar(x2 + offset, f1_scores, width, 
               label=model_name, 
               color=model_colors[model_name], 
               alpha=0.8,
               edgecolor='black',
               linewidth=0.5)
    
    ax2.set_xlabel('Class (sorted by Class ID)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax2.set_title('F1 Score Comparison - Sorted by Class ID', fontsize=14, fontweight='bold')
    ax2.set_xticks(x2)
    ax2.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in zip(tissue_names_by_id, sorted_by_id)], 
                        rotation=90, ha='center', fontsize=8)
    ax2.set_ylim(0, 1)
    ax2.legend(loc='upper right', fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
    ax2.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ 没有加载到模型数据")

## 7. 图表2：True vs Predicted Class Prevalence 对比

In [ ]:
if len(model_data) > 0:
    # 创建图表
    n_models = len(model_data)
    fig, axes = plt.subplots(n_models, 1, figsize=(20, 8*n_models))
    
    # 如果只有一个模型，axes不是数组
    if n_models == 1:
        axes = [axes]
    
    fig.suptitle('True vs Predicted Class Prevalence - Model Comparison', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    # 对每个模型创建一个子图
    for idx, (model_name, df) in enumerate(model_data.items()):
        ax = axes[idx]
        
        # 按class_id排序
        df_sorted = df.sort_values('class_id')
        
        x = np.arange(len(df_sorted))
        width = 0.35
        
        # 绘制真实prevalence和预测prevalence
        bars1 = ax.bar(x - width/2, df_sorted['prevalence'], width, 
                      label='True Prevalence', 
                      color='#1f77b4', 
                      alpha=0.8,
                      edgecolor='black',
                      linewidth=0.5)
        
        bars2 = ax.bar(x + width/2, df_sorted['predicted_prevalence'], width, 
                      label='Predicted Prevalence', 
                      color=model_colors[model_name], 
                      alpha=0.8,
                      edgecolor='black',
                      linewidth=0.5)
        
        ax.set_xlabel('Class', fontsize=12, fontweight='bold')
        ax.set_ylabel('Prevalence (fraction)', fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name} - True vs Predicted Prevalence', 
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in 
                           zip(df_sorted['tissue_name'], df_sorted['class_id'])], 
                          rotation=90, ha='center', fontsize=8)
        ax.set_yscale('log')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        
        # 添加连接线来显示差异
        for i in range(len(df_sorted)):
            true_val = df_sorted.iloc[i]['prevalence']
            pred_val = df_sorted.iloc[i]['predicted_prevalence']
            if true_val > 0 and pred_val > 0:
                ax.plot([i-width/2, i+width/2], [true_val, pred_val], 
                       'k--', alpha=0.3, linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ 没有加载到模型数据")

## 8. 统计汇总表

In [ ]:
# 创建模型性能汇总表
if len(model_data) > 0:
    summary_data = []
    
    for model_name, df in model_data.items():
        summary = {
            'Model': model_name,
            'Mean F1': df['f1_score'].mean(),
            'Median F1': df['f1_score'].median(),
            'Std F1': df['f1_score'].std(),
            'Min F1': df['f1_score'].min(),
            'Max F1': df['f1_score'].max(),
            'Classes with F1>0.8': (df['f1_score'] >= 0.8).sum(),
            'Classes with F1>0.6': (df['f1_score'] >= 0.6).sum(),
            'Classes with F1=0': (df['f1_score'] == 0).sum(),
            'Mean Dice': df['dice_coefficient'].mean(),
            'Mean Precision': df['precision'].mean(),
            'Mean Recall': df['recall'].mean(),
        }
        summary_data.append(summary)
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n" + "="*100)
    print("📊 模型性能汇总表")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("="*100)
    
    # 可视化汇总
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Model Performance Summary', fontsize=16, fontweight='bold')
    
    # 平均F1对比
    ax = axes[0]
    bars = ax.bar(summary_df['Model'], summary_df['Mean F1'], 
                  color=[model_colors[m] for m in summary_df['Model']],
                  alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Mean F1 Score', fontweight='bold')
    ax.set_title('Average F1 Score', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # F1分布箱线图
    ax = axes[1]
    f1_data = [model_data[m]['f1_score'].values for m in summary_df['Model']]
    bp = ax.boxplot(f1_data, labels=summary_df['Model'], patch_artist=True)
    for patch, model in zip(bp['boxes'], summary_df['Model']):
        patch.set_facecolor(model_colors[model])
        patch.set_alpha(0.8)
    ax.set_ylabel('F1 Score', fontweight='bold')
    ax.set_title('F1 Score Distribution', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 性能等级分布
    ax = axes[2]
    categories = ['Excellent\n(F1≥0.8)', 'Good\n(F1≥0.6)', 'Failed\n(F1=0)']
    x = np.arange(len(categories))
    width = 0.8 / len(model_data)
    
    for i, (model_name, _) in enumerate(model_data.items()):
        counts = [
            summary_df[summary_df['Model'] == model_name]['Classes with F1>0.8'].values[0],
            summary_df[summary_df['Model'] == model_name]['Classes with F1>0.6'].values[0],
            summary_df[summary_df['Model'] == model_name]['Classes with F1=0'].values[0],
        ]
        offset = (i - len(model_data)/2 + 0.5) * width
        ax.bar(x + offset, counts, width, 
              label=model_name, 
              color=model_colors[model_name],
              alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel('Number of Classes', fontweight='bold')
    ax.set_title('Performance Category Distribution', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

else:
    print("❌ 没有加载到模型数据")

## 9. 完成

✅ 所有图表已生成并显示！

### 使用说明：

1. **修改baseline的labels路径**：在第2个代码单元格中，修改`models`字典中Baseline的labels文件路径

2. **添加更多模型**：在`models`字典中添加更多模型配置，例如：
   ```python
   'KAN': 'results/per_class_analysis_kan_bg_excl_TIMESTAMP/per_class_detailed_metrics.csv',
   ```

3. **运行所有单元格**：点击 `Cell` → `Run All`